# 08g: Alias Equivalence & Order Independence

**Phase 13.6.G Debug Notebook**

## Purpose

Demonstrate that `alias()` with deferred validation produces identical results to `define()` with dependency-ordered definitions.

## Key Invariant

**Alias pool is order-independent**: aliases can reference other aliases that haven't been defined yet.

**Define is order-dependent**: each define must only reference columns or previously-defined expressions.

Both approaches must produce **mathematically identical results**.

## Test Pattern

```
Aliases (any order)     vs    Defines (dependency order)
─────────────────────        ────────────────────────────
total = scaled + first        sliced = track_pt[1:-1]
scaled = Sum(sliced)*w        first = track_pt[0]
sliced = track_pt[1:-1]       weight = event_weight  
first = track_pt[0]           scaled = Sum(sliced)*weight
weight = event_weight         total = scaled + first
```

In [ ]:
import numpy as np
import ROOT
import sys
sys.path.insert(0, '..')

from RDataFrameDSL import DSLCompiler
from tests.generators.toy_nd import generate_nd_2d_root

# Generate test data
filename = generate_nd_2d_root(size='S', seed=42)
rdf = ROOT.RDataFrame("Events", filename)

schema = {
    'event_id': 'long',
    'n_tracks': 'int',
    'event_weight': 'double',
    'track_pt': 'RVec<double>',
    'track_eta': 'RVec<double>',
}

print(f"Loaded {rdf.Count().GetValue()} events")
print(f"Schema: {list(schema.keys())}")

## Section 1: Alias Order Independence

### Description
Define aliases in **reverse dependency order** (referencing undefined aliases).

### What we're testing
- Aliases can reference other aliases not yet defined
- Resolution happens at compile time, not definition time
- Final result is correct regardless of definition order

In [ ]:
# METHOD A: Aliases in REVERSE dependency order
dsl_alias = DSLCompiler(schema)

# Define aliases referencing things that don't exist yet!
dsl_alias.alias("total_A", "scaled_A + first_A")        # Uses scaled_A, first_A (undefined!)
dsl_alias.alias("scaled_A", "Sum(track_pt) * weight_A") # Uses weight_A (undefined!)
dsl_alias.alias("first_A", "track_pt[0]")
dsl_alias.alias("weight_A", "event_weight")

# Materialize via to_pandas (this triggers compilation)
df_A = dsl_alias.to_pandas(rdf, columns=['total_A'])

print("Alias method: aliases defined in reverse order, materialized via to_pandas")
print(f"Result shape: {df_A.shape}")

In [ ]:
# METHOD D: Defines in CORRECT dependency order
dsl_define = DSLCompiler(schema)

# Must define in order: leaves first, then intermediate, then final
dsl_define.define("first_D", "track_pt[0]")
dsl_define.define("weight_D", "event_weight")
dsl_define.define("scaled_D", "Sum(track_pt) * weight_D")
dsl_define.define("total_D", "scaled_D + first_D")

df_D = dsl_define.to_pandas(rdf, columns=['total_D'])

print("Define method: must be in dependency order")
print(f"Result shape: {df_D.shape}")

In [ ]:
# INVARIANCE CHECK
result_A = df_A['total_A'].values
result_D = df_D['total_D'].values

match = np.allclose(result_A, result_D, rtol=1e-10)

print(f"\n{'='*60}")
print(f"INVARIANCE CHECK: Alias vs Define")
print(f"{'='*60}")
print(f"Result A (alias):  {result_A[:5]}...")
print(f"Result D (define): {result_D[:5]}...")
print(f"Match: {'✓ PASSED' if match else '✗ FAILED'}")
print(f"Max difference: {np.max(np.abs(result_A - result_D))}")

assert match, "Alias vs Define mismatch!"

## Section 2: Multiple Alias Orderings

### Description
Test that ANY order of alias definitions produces the same result.

### What we're testing
- Permutation invariance of alias definitions

In [ ]:
def build_with_alias_order(order_name, alias_order):
    """Build DSL with aliases in specified order."""
    dsl = DSLCompiler(schema)
    
    aliases = {
        'total': ("total", "scaled + first"),
        'scaled': ("scaled", "Sum(track_pt) * weight"),
        'first': ("first", "track_pt[0]"),
        'weight': ("weight", "event_weight"),
    }
    
    for key in alias_order:
        name, expr = aliases[key]
        dsl.alias(name, expr)
    
    return dsl

# Test different orderings
orderings = [
    ("Forward", ['first', 'weight', 'scaled', 'total']),
    ("Reverse", ['total', 'scaled', 'first', 'weight']),
    ("Random1", ['weight', 'total', 'first', 'scaled']),
    ("Random2", ['scaled', 'weight', 'total', 'first']),
]

results = {}
for name, order in orderings:
    dsl = build_with_alias_order(name, order)
    df = dsl.to_pandas(rdf, columns=['total'])
    results[name] = df['total'].values
    print(f"{name}: {df['total'].values[:3]}...")

In [ ]:
# Check all orderings produce identical results
print(f"\n{'='*60}")
print("PERMUTATION INVARIANCE CHECK")
print(f"{'='*60}")

reference = results['Forward']
all_match = True
for name, values in results.items():
    match = np.allclose(reference, values, rtol=1e-10)
    status = '✓' if match else '✗'
    print(f"{status} {name} vs Forward: max_diff={np.max(np.abs(reference - values)):.2e}")
    all_match = all_match and match

print(f"\nAll orderings equivalent: {'✓ PASSED' if all_match else '✗ FAILED'}")
assert all_match, "Ordering invariance failed!"

## Section 3: Mixed Syntax Equivalence

### Description
Show that inline expressions, aliases, and defines all produce identical results.

### What we're testing
- `define("x", "expr")` == `alias("a", "expr"); define("x", "a")`
- Commutativity: `a * b` == `b * a`

In [ ]:
# Three equivalent ways to compute: Sum(track_pt) * event_weight + track_pt[0]

# Method 1: Single inline expression
dsl1 = DSLCompiler(schema)
dsl1.define("result", "Sum(track_pt) * event_weight + track_pt[0]")
df1 = dsl1.to_pandas(rdf, columns=['result'])

# Method 2: Partial alias
dsl2 = DSLCompiler(schema)
dsl2.alias("middle_sum", "Sum(track_pt)")
dsl2.alias("result", "middle_sum * event_weight + track_pt[0]")
df2 = dsl2.to_pandas(rdf, columns=['result'])

# Method 3: Full decomposition with aliases
dsl3 = DSLCompiler(schema)
dsl3.alias("middle_sum", "Sum(track_pt)")
dsl3.alias("weighted", "middle_sum * event_weight")
dsl3.alias("first", "track_pt[0]")
dsl3.alias("result", "weighted + first")
df3 = dsl3.to_pandas(rdf, columns=['result'])

# Method 4: Different operator order (commutativity)
dsl4 = DSLCompiler(schema)
dsl4.define("result", "event_weight * Sum(track_pt) + track_pt[0]")
df4 = dsl4.to_pandas(rdf, columns=['result'])

# Extract results
r1 = df1['result'].values
r2 = df2['result'].values
r3 = df3['result'].values
r4 = df4['result'].values

In [ ]:
print(f"{'='*60}")
print("SYNTAX EQUIVALENCE CHECK")
print(f"{'='*60}")
print(f"Method 1 (inline):      {r1[:3]}...")
print(f"Method 2 (partial):     {r2[:3]}...")
print(f"Method 3 (decomposed):  {r3[:3]}...")
print(f"Method 4 (commutative): {r4[:3]}...")
print()

checks = [
    ("Inline vs Partial", np.allclose(r1, r2, rtol=1e-10)),
    ("Inline vs Decomposed", np.allclose(r1, r3, rtol=1e-10)),
    ("Inline vs Commutative", np.allclose(r1, r4, rtol=1e-10)),
]

all_pass = True
for name, passed in checks:
    status = '✓' if passed else '✗'
    print(f"{status} {name}")
    all_pass = all_pass and passed

print(f"\nAll syntax forms equivalent: {'✓ PASSED' if all_pass else '✗ FAILED'}")
assert all_pass, "Syntax equivalence failed!"

## Section 4: Visual Comparison

Overlay histograms to visually confirm equivalence.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Plot 1: Alias vs Define (result_A and result_D are numpy arrays from Section 1)
ax1 = axes[0]
ax1.hist(result_A, bins=50, alpha=0.7, label='Alias (reverse order)', color='blue')
ax1.hist(result_D, bins=50, alpha=0.5, label='Define (dependency order)', color='red')
ax1.set_xlabel('Value')
ax1.set_ylabel('Count')
ax1.set_title('Alias vs Define')
ax1.legend()

# Plot 2: Different alias orderings (results dict from Section 2)
ax2 = axes[1]
colors = ['blue', 'red', 'green', 'orange']
for (name, values), color in zip(results.items(), colors):
    ax2.hist(values, bins=50, alpha=0.5, label=name, color=color)
ax2.set_xlabel('Value')
ax2.set_title('Alias Order Permutations')
ax2.legend()

# Plot 3: Syntax variations (r1, r3, r4 from Section 3)
ax3 = axes[2]
ax3.hist(r1, bins=50, alpha=0.7, label='Inline', color='blue')
ax3.hist(r3, bins=50, alpha=0.5, label='Decomposed', color='green')
ax3.hist(r4, bins=50, alpha=0.3, label='Commutative', color='red')
ax3.set_xlabel('Value')
ax3.set_title('Syntax Variations')
ax3.legend()

plt.tight_layout()
plt.savefig('/tmp/alias_equivalence.png', dpi=100)
plt.show()

print("\n✓ Visual confirmation: All histograms perfectly overlap")

## Summary

### Demonstrated Invariants

| Property | Status |
|----------|--------|
| Alias order independence | ✓ |
| Alias vs Define equivalence | ✓ |
| Permutation invariance | ✓ |
| Syntax equivalence | ✓ |
| Commutativity (a*b == b*a) | ✓ |

### Key Takeaway

**Aliases provide TTree::SetAlias-style convenience** - define expressions in any order, reference before definition.

**Defines require dependency order** - but can reference previously-defined expressions.

**Both produce identical numerical results** - verified by invariance tests.

In [ ]:
print("="*60)
print("08g_alias_equivalence.ipynb: ALL CHECKS PASSED")
print("="*60)